# CSA — full Karpathy TRAIN/val ROI preparation (CPU)

Accelerator NONE + Internet ON. Add FOUR Datasets:
`mscoco-2014` (vuthetam), `objectdetectionecache`, `visual-cache`, `csa-real-v126-lemma-fixed` (ducanh2403).
No Secret, checkpoint, bank130, spaCy, CLIP encoding or training. Uses existing rawCLIPpatches+predictedboxes.
Prepares113287TRAIN+5000val, excludes test. Writes119pairedNPZ/JSONshards. CPUcap6h is notETA; stops early if first-shard projection cannot fit cap.
Keep entire output folder; review ZIP excludes shards. Partial completion is not training-ready; no automatic resume.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
COMMIT='165dd753fde22746df3a2fd06824a3b6de720218'
HASHES={'real_bank_data.py': '00e7110105ec98a5ef41969f9e51074815a42a9b69deac90b2614e9679d2bcf9', 'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'REAL_BANK_INPUT_PACKAGE.json': 'c60902e04ea15cf90f652e384937bb317e288753f13b8ede6ab5ef89f8d97129', 'PRE_FULL_CHECK_PROTOCOL.md': '3e11fb2dde0e28c2b744a469bdf52e850f01c3696fddad4e78fed0e0eec716f3', 'prepare_full_regions.py': 'c6faae850f023374aad78dfaec9a5e33a5722f5c4cf78832697ec4f12b48c33b', 'prepare_real_assets.py': '44a22d92acd239bdc7a1edbe30984daf0853ac8bc196969dd48b19b78df1cc7b', 'audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959'}
OUTPUT=Path('/kaggle/working/csa_full_roi_karpathy')
PREPARED_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-real-v126-lemma-fixed')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use a fresh session.')
if shutil.disk_usage('/kaggle/working').free<8*1024**3:raise RuntimeError('Insufficient free disk; preserve existing files.')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='csa_full_roi_karpathy-'))
LOG=OUTPUT/'notebook.log'
for n,h in HASHES.items():
    data=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{n}',timeout=90).read()
    if hashlib.sha256(data).hexdigest()!=h:raise ValueError('Changed source: '+n)
    target=SOURCE/n;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')
print('Source pinned:',COMMIT,flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Worker active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Worker wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:

    os.environ['CUDA_VISIBLE_DEVICES']=''
    CONFIG=SOURCE/'full_roi_config.json'
    config=dict(dataset_json='/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json',
        detections='/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json',
        visual_cache='/kaggle/input/datasets/ducanh2403/visual-cache',
        image_root='/kaggle/input/datasets/vuthetam/mscoco-2014/images',prepared_root=str(PREPARED_ROOT),output=str(OUTPUT))
    # Edit paths here only if your mounted Dataset paths differ.
    CONFIG.write_text(json.dumps(config,indent=2),encoding='utf-8')
    run([sys.executable,'-u',SOURCE/'prepare_full_regions.py','--config',CONFIG],timeout=21600)
    receipt=json.loads((OUTPUT/'receipt.json').read_text(encoding='utf-8'))
    if receipt['status']!='FULL_ROI_PREPARATION_COMPLETE':raise RuntimeError('Full ROI incomplete')

    complete=True
except Exception:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training_executed=False)),encoding='utf-8')
    review=Path(str(OUTPUT)+'_review.zip')
    globals_only={'source_receipt.json','input_receipt.json','progress.json','manifest.json','receipt.json',
                  'throughput.json','failure.txt','notebook.log','run_status.json'}
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.iterdir()):
            if p.is_file() and p.suffix in ('.json','.log','.txt') and ('roi'=='decode' or p.name in globals_only):z.write(p,p.name)
    print('SEND REVIEW:',review,flush=True)
    print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('COMPLETE' if complete else 'INCOMPLETE; preserve completed artifacts/logs.',flush=True)
